# OncoInsights — Module 5: SQL Analytics

Runs each query from `sql/queries.sql` against `data/processed/oncoinsights.db`
(SQLite) and explains what it demonstrates and what the result means.

**Schema:**
- `patients` — one row per patient, clinical fields + engineered features (RISK_SCORE, HIGH_RISK_FLAG, STAGE_GROUP, etc.)
- `mutations` — one row per mutation call, 40-gene LUAD driver panel
- `expression` — long format: `patientId`, `sampleId`, `hugoGeneSymbol`, `log2_expression`


In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../data/processed/oncoinsights.db")
conn = sqlite3.connect(DB_PATH)
pd.set_option("display.max_columns", 20)


## Q9. Reusable view — high-risk cohort
**Technique:** CREATE VIEW

Defines `high_risk_cohort` once so every future query (or the dashboard) can just
`SELECT * FROM high_risk_cohort` instead of repeating the `HIGH_RISK_FLAG = 1` filter everywhere.
This is the standard pattern for a reusable cohort definition in a data warehouse. DDL statements
(DROP/CREATE) run via `executescript`, since SQLite only allows one statement per `execute` call.

In [2]:
conn.executescript("""
DROP VIEW IF EXISTS high_risk_cohort;
CREATE VIEW high_risk_cohort AS
SELECT patientId, sampleId, AGE, SEX, STAGE_GROUP, RISK_SCORE,
    PANEL_MUTATION_COUNT, SURVIVAL_MONTHS, EVENT_OCCURRED
FROM patients WHERE HIGH_RISK_FLAG = 1;
""")

df = pd.read_sql("""
SELECT COUNT(*) AS n_high_risk, ROUND(AVG(AGE), 1) AS avg_age,
    ROUND(AVG(PANEL_MUTATION_COUNT),2) AS avg_mutation_count,
    ROUND(AVG(EVENT_OCCURRED) * 100,1) AS pct_deceased,
    ROUND(AVG(SURVIVAL_MONTHS), 1) AS avg_survival_months
FROM high_risk_cohort;
""", conn)
df

,n_high_risk,avg_age,avg_mutation_count,pct_deceased,avg_survival_months
0,126,69.8,9.12,46.0,25.8


**Result:** the high-risk cohort (126 patients) has a 46% mortality rate vs the cohort-wide 36.2% in Q1 — an early signal that the risk score is doing real stratification work (confirmed formally by the log-rank test in Module 6).

## Q1. Overview KPIs
**Technique:** Aggregation

Headline numbers for the dashboard's Overview tab: cohort size, average age, average risk score, count of high-risk patients, and overall mortality rate. This is the query a stakeholder would run first to get oriented.

In [3]:
df = pd.read_sql("""SELECT
    COUNT(*) AS patient_count,
    ROUND(AVG(AGE), 1) AS avg_age,
    ROUND(AVG(RISK_SCORE), 3) AS avg_risk_score,
    SUM(HIGH_RISK_FLAG) AS high_risk_patients,
    ROUND(AVG(EVENT_OCCURRED) * 100, 1) AS pct_deceased
FROM patients;""", conn)
df

,patient_count,avg_age,avg_risk_score,high_risk_patients,pct_deceased
0,503,65.2,0.072,126,36.2


**Result:** 503 patients, average age ~65, ~25% flagged high-risk (matches the engineered threshold), 36% overall mortality in this cohort's follow-up window.

## Q2. Patient count & average age by stage
**Technique:** Aggregation + GROUP BY

Basic stratified aggregation — the first thing an analyst does with a new stage field: does age or risk score trend with disease stage?

In [4]:
df = pd.read_sql("""SELECT
    STAGE_GROUP, COUNT(*) AS patient_count, ROUND(AVG(AGE), 1) AS avg_age,
    ROUND(AVG(RISK_SCORE), 3) AS avg_risk_score
FROM patients
GROUP BY STAGE_GROUP
ORDER BY CASE STAGE_GROUP WHEN 'I' THEN 1 WHEN 'II' THEN 2 WHEN 'III' THEN 3 WHEN 'IV' THEN 4 END;""", conn)
df

,STAGE_GROUP,patient_count,avg_age,avg_risk_score
0,I,274,65.7,-0.004
1,II,122,65.0,0.139
2,III,80,65.7,0.201
3,IV,27,60.5,0.163


**Result:** average risk score rises from Stage I (-0.004) through Stage III (0.201) — expected, since stage is one of the three RISK_SCORE inputs. Stage IV patients are notably younger on average (60.5), consistent with more aggressive disease occurring in younger patients.

## Q3. Average EGFR expression by stage
**Technique:** Aggregation + JOIN

Joins the long-format expression table to patients to answer a specific clinical question: does EGFR expression track with tumor stage?

In [5]:
df = pd.read_sql("""SELECT p.STAGE_GROUP, COUNT(*) AS n_samples, ROUND(AVG(e.log2_expression), 3) AS avg_egfr_log2_expr
FROM expression e JOIN patients p ON p.patientId = e.patientId
WHERE e.hugoGeneSymbol = 'EGFR'
GROUP BY p.STAGE_GROUP ORDER BY avg_egfr_log2_expr DESC;""", conn)
df

,STAGE_GROUP,n_samples,avg_egfr_log2_expr
0,III,80,10.117
1,II,121,9.912
2,IV,27,9.911
3,I,271,9.823


**Result:** EGFR expression is fairly flat across stages (~9.8-10.1 log2 units) — no strong stage-dependence, which makes sense: EGFR activation is an early oncogenic driver event, not something that should keep changing as the tumor progresses through stages.

## Q4. Average age by TP53 mutation status
**Technique:** Aggregation + correlated subquery

Uses a subquery to derive mutation status inline, then aggregates age by that derived group.

In [6]:
df = pd.read_sql("""SELECT
    CASE WHEN p.patientId IN (SELECT DISTINCT patientId FROM mutations WHERE hugoGeneSymbol = 'TP53')
         THEN 'TP53 Mutant' ELSE 'TP53 Wild-type' END AS tp53_status,
    COUNT(*) AS patient_count, ROUND(AVG(p.AGE), 1) AS avg_age
FROM patients p GROUP BY tp53_status;""", conn)
df

,tp53_status,patient_count,avg_age
0,TP53 Mutant,262,63.8
1,TP53 Wild-type,241,66.8


**Result:** TP53-mutant patients are ~3 years younger on average (63.8 vs 66.8) — consistent with TP53 loss being associated with more biologically aggressive tumors that can present earlier.

## Q5. Top 10 most frequently mutated genes
**Technique:** Aggregation + ORDER BY + LIMIT

The classic 'what are our most common mutations' query — foundational for any mutation-analytics tab.

In [7]:
df = pd.read_sql("""SELECT hugoGeneSymbol, COUNT(DISTINCT patientId) AS patients_mutated,
    ROUND(100.0 * COUNT(DISTINCT patientId) / (SELECT COUNT(*) FROM patients), 1) AS pct_of_cohort
FROM mutations GROUP BY hugoGeneSymbol ORDER BY patients_mutated DESC LIMIT 10;""", conn)
df

,hugoGeneSymbol,patients_mutated,pct_of_cohort
0,TP53,262,52.1
1,LRP1B,174,34.6
2,ZFHX4,167,33.2
3,KRAS,153,30.4
4,SPTA1,135,26.8
5,XIRP2,127,25.2
6,COL11A1,106,21.1
7,KEAP1,94,18.7
8,STK11,74,14.7
9,EGFR,65,12.9


**Result:** TP53 leads at 52% of the cohort (matches published TCGA-LUAD rates), followed by LRP1B and ZFHX4 (large genes with high background/passenger mutation rates) and KRAS at 30%.

## Q6. Rank patients by risk score within stage
**Technique:** Window function — RANK()

Answers 'who are the highest-risk patients within each stage group?' — exactly the kind of ranked list a clinical team would want for triage, without collapsing the stage groups together.

In [8]:
df = pd.read_sql("""SELECT patientId, STAGE_GROUP, RISK_SCORE,
    RANK() OVER (PARTITION BY STAGE_GROUP ORDER BY RISK_SCORE DESC) AS risk_rank_in_stage
FROM patients ORDER BY STAGE_GROUP, risk_rank_in_stage LIMIT 20;""", conn)
df

,patientId,STAGE_GROUP,RISK_SCORE,risk_rank_in_stage
0,TCGA-05-4382,I,2.275474,1
1,TCGA-69-7979,I,1.530038,2
2,TCGA-78-7155,I,1.439452,3
3,TCGA-55-8511,I,1.067915,4
4,TCGA-38-4631,I,1.037719,5
5,TCGA-05-4405,I,0.993607,6
6,TCGA-78-8662,I,0.986523,7
7,TCGA-67-3771,I,0.979690,8
8,TCGA-05-4410,I,0.944772,9
9,TCGA-44-5644,I,0.926133,10


**Result:** PARTITION BY resets the ranking for each stage group independently, so the #1-ranked Stage I patient and #1-ranked Stage IV patient are each the highest-risk *within their own stage*, not globally.

## Q7. Risk quartile & percentile
**Technique:** Window functions — NTILE(), PERCENT_RANK()

Two more window functions for cohort segmentation: NTILE(4) buckets patients into quartiles, PERCENT_RANK() gives a continuous 0-100 percentile — both common dashboard filter inputs.

In [9]:
df = pd.read_sql("""SELECT patientId, RISK_SCORE,
    NTILE(4) OVER (ORDER BY RISK_SCORE) AS risk_quartile,
    ROUND(PERCENT_RANK() OVER (ORDER BY RISK_SCORE) * 100, 1) AS risk_percentile
FROM patients ORDER BY RISK_SCORE DESC LIMIT 20;""", conn)
df

,patientId,RISK_SCORE,risk_quartile,risk_percentile
0,TCGA-05-4382,2.275474,4,100.0
1,TCGA-55-A490,2.154913,4,99.8
2,TCGA-95-7567,1.537091,4,99.6
3,TCGA-69-7979,1.530038,4,99.4
4,TCGA-55-7994,1.513979,4,99.2
5,TCGA-78-7155,1.439452,4,99.0
6,TCGA-05-4427,1.344364,4,98.8
7,TCGA-95-7039,1.221221,4,98.6
8,TCGA-55-7907,1.184193,4,98.4
9,TCGA-05-4424,1.181832,4,98.2


**Result:** the highest-risk patient sits at the 100th percentile with a RISK_SCORE of 2.28 — over 2 standard deviations above the cohort mean of ~0.07.

## Q8. Avg mutation count by stage
**Technique:** Multi-step CTE

A genuine two-step CTE: first aggregate mutation counts per patient (`patient_mutation_counts`), then LEFT JOIN that onto every patient (including zero-mutation patients, via COALESCE) before the final stage-level aggregation. Demonstrates why CTEs beat nested subqueries for readability.

In [10]:
df = pd.read_sql("""WITH patient_mutation_counts AS (
    SELECT patientId, COUNT(*) AS mut_count FROM mutations GROUP BY patientId
),
patients_with_counts AS (
    SELECT p.patientId, p.STAGE_GROUP, COALESCE(pmc.mut_count, 0) AS mut_count
    FROM patients p LEFT JOIN patient_mutation_counts pmc ON pmc.patientId = p.patientId
)
SELECT STAGE_GROUP, COUNT(*) AS patient_count, ROUND(AVG(mut_count), 2) AS avg_panel_mutation_count
FROM patients_with_counts GROUP BY STAGE_GROUP
ORDER BY CASE STAGE_GROUP WHEN 'I' THEN 1 WHEN 'II' THEN 2 WHEN 'III' THEN 3 WHEN 'IV' THEN 4 END;""", conn)
df

,STAGE_GROUP,patient_count,avg_panel_mutation_count
0,I,274,4.85
1,II,122,5.48
2,III,80,4.90
3,IV,27,5.07


**Result:** panel mutation count is fairly flat across stages (~4.9-5.5) — the LEFT JOIN + COALESCE matters here, because without it, the 29 patients with zero panel mutations would silently drop out of the average instead of correctly pulling it down.

## Q10. Survival outcomes by risk group
**Technique:** Aggregation, business-relevant

The business-facing version of the Kaplan-Meier result from Module 6 — a single aggregated table comparing high-risk vs everyone-else on average survival months and mortality rate.

In [11]:
df = pd.read_sql("""SELECT
    CASE WHEN HIGH_RISK_FLAG = 1 THEN 'High risk' ELSE 'Low/medium risk' END AS risk_group,
    COUNT(*) AS patient_count, ROUND(AVG(SURVIVAL_MONTHS), 1) AS avg_survival_months,
    ROUND(AVG(EVENT_OCCURRED) * 100, 1) AS pct_deceased
FROM patients GROUP BY risk_group;""", conn)
df

,risk_group,patient_count,avg_survival_months,pct_deceased
0,High risk,126,25.8,46.0
1,Low/medium risk,377,31.1,32.9


**Result:** high-risk patients average 25.8 months of follow-up survival vs 31.1 for low/medium-risk, with a higher mortality rate (46.0% vs 32.9%) — directionally consistent with the log-rank test result.

## Q11. Event rate by EGFR expression quartile
**Technique:** Aggregation + JOIN via engineered feature

Uses the `EGFR_EXPR_QUARTILE` feature engineered in Module 4 directly — this is the payoff of engineering that column instead of re-deriving quartiles in every query.

In [12]:
df = pd.read_sql("""SELECT EGFR_EXPR_QUARTILE, COUNT(*) AS patient_count,
    ROUND(AVG(EVENT_OCCURRED) * 100, 1) AS pct_deceased,
    ROUND(AVG(SURVIVAL_MONTHS), 1) AS avg_survival_months
FROM patients WHERE EGFR_EXPR_QUARTILE IS NOT NULL
GROUP BY EGFR_EXPR_QUARTILE ORDER BY EGFR_EXPR_QUARTILE;""", conn)
df

,EGFR_EXPR_QUARTILE,patient_count,pct_deceased,avg_survival_months
0,Q1 (low),125,37.6,34.5
1,Q2,125,36.8,29.7
2,Q3,124,31.5,27.6
3,Q4 (high),125,39.2,27.4


**Result:** mortality rate is fairly flat across EGFR quartiles (31.5%-39.2%) with no clean monotonic trend — EGFR expression level alone isn't a strong survival signal in this cohort, which is a legitimate (if less exciting) finding worth reporting honestly.

## Q12. Top 5 mutated genes per stage, ranked
**Technique:** Multi-step CTE + window function

Combines both techniques: a CTE aggregates mutation counts per gene per stage, then a second CTE ranks genes within each stage, and the outer query filters to the top 5 per stage.

In [13]:
df = pd.read_sql("""WITH gene_stage_counts AS (
    SELECT m.hugoGeneSymbol, p.STAGE_GROUP, COUNT(DISTINCT m.patientId) AS n_mutated
    FROM mutations m JOIN patients p ON p.patientId = m.patientId
    GROUP BY m.hugoGeneSymbol, p.STAGE_GROUP
),
ranked AS (
    SELECT STAGE_GROUP, hugoGeneSymbol, n_mutated,
        RANK() OVER (PARTITION BY STAGE_GROUP ORDER BY n_mutated DESC) AS rank_in_stage
    FROM gene_stage_counts
)
SELECT * FROM ranked WHERE rank_in_stage <= 5 ORDER BY STAGE_GROUP, rank_in_stage;""", conn)
df

,STAGE_GROUP,hugoGeneSymbol,n_mutated,rank_in_stage
0,I,TP53,138,1
1,I,LRP1B,96,2
2,I,ZFHX4,95,3
3,I,KRAS,77,4
4,I,SPTA1,71,5
5,II,TP53,65,1
6,II,ZFHX4,43,2
7,II,KRAS,40,3
8,II,LRP1B,39,4
9,II,SPTA1,39,4


**Result:** TP53 is the #1 mutated gene in every stage group — the mutation landscape's overall shape (TP53 > LRP1B/ZFHX4 > KRAS) is consistent across stages, meaning stage doesn't reshuffle which genes dominate, just how many patients are affected.

In [14]:
conn.close()